In [1]:
import pandas as pd, numpy as np, gc
from pathlib import Path

PROC = Path('../data/processed')

def mem_mb(df):
    return df.memory_usage(deep=True).sum() / 1024**2

def downcast(df):
    for c in df.select_dtypes('float').columns:
        df[c] = pd.to_numeric(df[c], downcast='float')
    for c in df.select_dtypes('integer').columns:
        df[c] = pd.to_numeric(df[c], downcast='integer')
    return df

train = pd.read_parquet(PROC / 'application_train_reduced.parquet')
test  = pd.read_parquet(PROC / 'application_test_reduced.parquet')

print(f"train: {train.shape} | {mem_mb(train):,.1f} MB")
print(f"test : {test.shape} | {mem_mb(test):,.1f} MB")

# Claves únicas (requisito para validate='1:1' en el merge)
assert train['SK_ID_CURR'].is_unique, "SK_ID_CURR duplicado en train!"
assert test['SK_ID_CURR'].is_unique,  "SK_ID_CURR duplicado en test!"
print("SK_ID_CURR único en ambas tablas: OK")

# Diferencia de columnas base (debería ser solo TARGET)
print("Solo en train:", sorted(set(train.columns) - set(test.columns)))
print("Solo en test :", sorted(set(test.columns) - set(train.columns)))

# Lista COMPLETA de columnas de train (para decidir los drops con precisión)
print("\n--- COLUMNAS application_train_reduced ---")
for i, c in enumerate(train.columns):
    print(f"{i:3d}  {c:38s} {str(train[c].dtype)}")

train: (307511, 90) | 311.5 MB
test : (48744, 89) | 49.3 MB
SK_ID_CURR único en ambas tablas: OK
Solo en train: ['TARGET']
Solo en test : []

--- COLUMNAS application_train_reduced ---
  0  SK_ID_CURR                             int32
  1  TARGET                                 int8
  2  NAME_CONTRACT_TYPE                     object
  3  CODE_GENDER                            object
  4  FLAG_OWN_CAR                           object
  5  FLAG_OWN_REALTY                        object
  6  CNT_CHILDREN                           int8
  7  AMT_INCOME_TOTAL                       float64
  8  AMT_CREDIT                             float32
  9  AMT_ANNUITY                            float32
 10  AMT_GOODS_PRICE                        float32
 11  NAME_TYPE_SUITE                        object
 12  NAME_INCOME_TYPE                       object
 13  NAME_EDUCATION_TYPE                    object
 14  NAME_FAMILY_STATUS                     object
 15  NAME_HOUSING_TYPE                      object


In [2]:
agg_tables = [
    ('bureau_master.parquet',           'HAS_BUREAU'),
    ('previous_app_aggregated.parquet', 'HAS_PREV'),
    ('pos_cash_aggregated.parquet',     'HAS_POS'),
    ('credit_card_aggregated.parquet',  'HAS_CC'),
    ('installments_aggregated.parquet', 'HAS_INST'),
]

def merge_all(base, name):
    base = base.copy()                      # no mutar el original
    print(f"\n=== Construyendo master_{name} ===")
    print(f"  base{'':37s} shape={base.shape}")
    for fname, flag in agg_tables:
        agg = pd.read_parquet(PROC / fname)
        dups = agg['SK_ID_CURR'].duplicated().sum()
        if dups:
            print(f"  ⚠️  {fname}: {dups} SK_ID_CURR DUPLICADOS — la agregación NO es 1 fila/cliente!")
        # flag de presencia ANTES del join (la ausencia es informativa)
        base[flag] = base['SK_ID_CURR'].isin(agg['SK_ID_CURR']).astype('int8')
        c0 = base.shape[1]
        base = base.merge(agg, on='SK_ID_CURR', how='left', validate='1:1')
        cov = base[flag].mean() * 100
        print(f"  + {fname:33s} (+{base.shape[1]-c0:2d} cols) {flag:11s}={cov:5.2f}%  shape={base.shape}")
        del agg; gc.collect()
    collide = [c for c in base.columns if c.endswith(('_x', '_y'))]
    print("  Colisiones de nombres _x/_y:", collide if collide else "ninguna ✓")
    print(f"  master_{name}: {base.shape} | {mem_mb(base):,.1f} MB")
    return base

master_train = merge_all(train, 'train')
master_test  = merge_all(test,  'test')
del train, test; gc.collect()


=== Construyendo master_train ===
  base                                      shape=(307511, 90)
  + bureau_master.parquet             (+48 cols) HAS_BUREAU =85.69%  shape=(307511, 139)
  + previous_app_aggregated.parquet   (+37 cols) HAS_PREV   =94.65%  shape=(307511, 177)
  + pos_cash_aggregated.parquet       (+25 cols) HAS_POS    =94.12%  shape=(307511, 203)
  + credit_card_aggregated.parquet    (+31 cols) HAS_CC     =28.26%  shape=(307511, 235)
  + installments_aggregated.parquet   (+23 cols) HAS_INST   =94.84%  shape=(307511, 259)
  Colisiones de nombres _x/_y: ninguna ✓
  master_train: (307511, 259) | 660.1 MB

=== Construyendo master_test ===
  base                                      shape=(48744, 89)
  + bureau_master.parquet             (+48 cols) HAS_BUREAU =86.82%  shape=(48744, 138)
  + previous_app_aggregated.parquet   (+37 cols) HAS_PREV   =98.06%  shape=(48744, 176)
  + pos_cash_aggregated.parquet       (+25 cols) HAS_POS    =98.08%  shape=(48744, 202)
  + credit_card

0

In [3]:
# Paridad de columnas (esperado: solo TARGET sobra en train)
print("Solo en master_train (esperado ['TARGET']):",
      sorted(set(master_train.columns) - set(master_test.columns)))
print("Solo en master_test :",
      sorted(set(master_test.columns) - set(master_train.columns)))

# --- Candidatos a drop por redundancia (NO se dropea todavía) ---
print("\nColumnas que empiezan con 'YEARS_' (separar housing _AVG de las derivadas de DAYS):")
for c in [c for c in master_train.columns if c.startswith('YEARS_')]:
    print("   ", c)

print("\nPresencia de columnas clave para el drop:")
for col in ['FLAG_EMP_PHONE', 'DAYS_EMPLOYED_ANOM',
            'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY']:
    print(f"   {col:32s} -> {col in master_train.columns}")

print(f"\nmaster_train: {master_train.shape} | {mem_mb(master_train):,.1f} MB")
print(f"master_test : {master_test.shape} | {mem_mb(master_test):,.1f} MB")

Solo en master_train (esperado ['TARGET']): ['TARGET']
Solo en master_test : []

Columnas que empiezan con 'YEARS_' (separar housing _AVG de las derivadas de DAYS):
    YEARS_BEGINEXPLUATATION_AVG
    YEARS_BUILD_AVG
    YEARS_SINCE_REGISTRATION
    YEARS_SINCE_ID_PUBLISH
    YEARS_SINCE_PHONE_CHANGE

Presencia de columnas clave para el drop:
   FLAG_EMP_PHONE                   -> True
   DAYS_EMPLOYED_ANOM               -> True
   REGION_RATING_CLIENT             -> True
   REGION_RATING_CLIENT_W_CITY      -> True

master_train: (307511, 259) | 660.1 MB
master_test : (48744, 258) | 104.6 MB


In [4]:
REDUNDANT_DROPS = [
    'AGE_YEARS',
    'EMPLOYMENT_YEARS',
    'YEARS_SINCE_REGISTRATION',
    'YEARS_SINCE_ID_PUBLISH',
    'YEARS_SINCE_PHONE_CHANGE',
    'FLAG_EMP_PHONE',
]

# Sanity: que existan los que voy a dropear y que NO arrastre las housing por error
for c in REDUNDANT_DROPS:
    assert c in master_train.columns, f"No existe la columna a dropear: {c}"
for keep in ['YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG']:
    assert keep in master_train.columns, f"Se perdió housing: {keep}"

master_train = master_train.drop(columns=REDUNDANT_DROPS)
master_test  = master_test.drop(columns=REDUNDANT_DROPS)
print("Dropeadas:", REDUNDANT_DROPS)
print("master_train:", master_train.shape, "| master_test:", master_test.shape)

# Paridad final (esperado: solo TARGET sobra en train)
diff_tr = sorted(set(master_train.columns) - set(master_test.columns))
diff_te = sorted(set(master_test.columns) - set(master_train.columns))
print("Solo en train:", diff_tr, "| Solo en test:", diff_te)
assert diff_tr == ['TARGET'] and diff_te == [], "¡Desalineación train/test!"
print("Paridad train/test: OK")

Dropeadas: ['AGE_YEARS', 'EMPLOYMENT_YEARS', 'YEARS_SINCE_REGISTRATION', 'YEARS_SINCE_ID_PUBLISH', 'YEARS_SINCE_PHONE_CHANGE', 'FLAG_EMP_PHONE']
master_train: (307511, 253) | master_test: (48744, 252)
Solo en train: ['TARGET'] | Solo en test: []
Paridad train/test: OK


In [5]:
print(f"Antes downcast  -> train: {mem_mb(master_train):,.1f} MB | test: {mem_mb(master_test):,.1f} MB")
master_train = downcast(master_train)
master_test  = downcast(master_test)
gc.collect()
print(f"Después downcast-> train: {mem_mb(master_train):,.1f} MB | test: {mem_mb(master_test):,.1f} MB")

# Asegurar MISMO ORDEN de columnas en test (= train sin TARGET)
feat_cols = [c for c in master_train.columns if c != 'TARGET']
master_test = master_test[feat_cols]

master_train.to_parquet(PROC / 'master_train.parquet', index=False)
master_test.to_parquet(PROC / 'master_test.parquet',  index=False)
print("Guardados: master_train.parquet, master_test.parquet")

# Verificación por reload (que el parquet roundtripee bien)
chk_tr = pd.read_parquet(PROC / 'master_train.parquet')
chk_te = pd.read_parquet(PROC / 'master_test.parquet')
print(f"reload -> train: {chk_tr.shape} | test: {chk_te.shape}")
assert list(chk_te.columns) == [c for c in chk_tr.columns if c != 'TARGET'], "orden de columnas no calza"
print("Orden de columnas train==test (sin TARGET): OK")
print("Cobertura HAS_* (train):",
      {c: round(chk_tr[c].mean()*100, 2) for c in ['HAS_BUREAU','HAS_PREV','HAS_POS','HAS_CC','HAS_INST']})
print("TARGET:", chk_tr['TARGET'].dtype, "| positivos:", round(chk_tr['TARGET'].mean()*100, 2), "%")
del chk_tr, chk_te; gc.collect()

Antes downcast  -> train: 654.0 MB | test: 103.6 MB
Después downcast-> train: 553.1 MB | test: 87.6 MB
Guardados: master_train.parquet, master_test.parquet
reload -> train: (307511, 253) | test: (48744, 252)
Orden de columnas train==test (sin TARGET): OK
Cobertura HAS_* (train): {'HAS_BUREAU': np.float64(85.69), 'HAS_PREV': np.float64(94.65), 'HAS_POS': np.float64(94.12), 'HAS_CC': np.float64(28.26), 'HAS_INST': np.float64(94.84)}
TARGET: int8 | positivos: 8.07 %


0